# LangGraph Short-Term Memory with Azure Cosmos DB

### Introduction

This notebook demonstrates how to add **short-term memory** (conversation persistence) to a LangGraph agent using **Azure Cosmos DB** as the checkpointer backend.

Short-term memory in LangGraph is implemented via **checkpointers**, which save a snapshot of the graph state at every step. When you assign a `thread_id` to a conversation, the checkpointer persists messages across invocations — enabling multi-turn conversations where the agent remembers what was said earlier.

While `InMemorySaver` works for prototyping, it loses all state when the process restarts. For production workloads you need a durable backend. Azure Cosmos DB provides global distribution, serverless pricing, automatic indexing, and Microsoft Entra ID support for secure, keyless authentication.

```
User ──► LangGraph Agent ──► Self-hosted LLM (vLLM on ACA)
              │
              ▼
        CosmosDB Checkpointer
        (persists conversation state per thread)
```

### Prerequisites

- Completion of steps in `010_langchain_basics.ipynb` where we deployed an Azure Foundry with an LLM model.
- An Azure Cosmos DB account deployed (see the `infra` folder) as part of `010_langchain_basics.ipynb`.

### Getting the LLM Endpoint and Cosmos DB Details

To use the LLM model with `langchain`, we first need to retrieve the self-hosted LLM endpoint and the Cosmos DB connection details from the Terraform outputs. You can do this by running the following command.


In [2]:
foundry_endpoint = ! terraform -chdir=./infra output -raw foundry_endpoint
foundry_endpoint = foundry_endpoint.n
print("Foundry Endpoint:", foundry_endpoint)

foundry_api_key = ! terraform -chdir=./infra output -raw foundry_api_key
foundry_api_key = foundry_api_key.n
print("Foundry API Key:", f"{foundry_api_key[-10:]}...")  # Print only the last 10 characters for security

llm_model_deployment_name = ! terraform -chdir=./infra output -raw llm_model_deployment_name
llm_model_deployment_name = llm_model_deployment_name.n
print("LLM Model Deployment Name (ChatGPT):", llm_model_deployment_name)

cosmosdb_endpoint = ! terraform -chdir=./infra output -raw cosmosdb_endpoint
cosmosdb_endpoint = cosmosdb_endpoint.n
print("CosmosDB Endpoint:", cosmosdb_endpoint)

cosmosdb_key = ! terraform -chdir=./infra output -raw cosmosdb_key
cosmosdb_key = cosmosdb_key.n
print("CosmosDB Key:", cosmosdb_key[:10] + "...")

Foundry Endpoint: https://foundry-400.cognitiveservices.azure.com/
Foundry API Key: AAACOGYQ3t...
LLM Model Deployment Name (ChatGPT): gpt-5.4
CosmosDB Endpoint: https://cosmosdb-memory-400.documents.azure.com:443/
CosmosDB Key: DR4yo3JZQ6...


### 1. Set Up the LLM Model

Then we can test the endpoint and key by making a simple API call to the LLM model.

In [5]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

model = ChatOpenAI(
    base_url=f"{foundry_endpoint}/openai/v1",
    api_key=foundry_api_key,
    model=llm_model_deployment_name,
    streaming=True,
    max_completion_tokens=512
)

response = model.stream([HumanMessage(content="Tell me about yourself.")])

for chunk in response:
    print(chunk.content, end="", flush=True)

I’m ChatGPT, an AI assistant created by OpenAI.

I can help with things like:
- answering questions
- explaining concepts
- writing and editing
- brainstorming ideas
- coding help
- summarizing information
- tutoring and step-by-step problem solving

A few important notes about me:
- I don’t have feelings, beliefs, or personal experiences.
- I generate responses based on patterns in data I was trained on.
- I can be useful, but I can also make mistakes, so it’s good to verify important information.
- I don’t know everything in real time unless you provide context or I have access to tools that supply it.

You can treat me like a general-purpose thinking and writing partner. If you want, I can also tell you:
- what I’m good at
- what my limits are
- how to get better answers from me

### 2. Create the CosmosDB Checkpointer

The `CosmosDBSaverSync` checkpointer from `langchain-azure-cosmosdb` persists LangGraph state (messages, tool calls, etc.) into Azure Cosmos DB.

This is the key ingredient for **short-term memory**: every message in a conversation thread is saved to CosmosDB, so the agent can recall earlier messages in the same thread.

In [9]:
from langchain_azure_cosmosdb import CosmosDBSaverSync

checkpointer = CosmosDBSaverSync(
    database_name="langgraph-memory",
    container_name="checkpoints",
    endpoint=cosmosdb_endpoint,
    key=cosmosdb_key,
)

### 3. Create an Agent with Memory

Compile a LangGraph agent with the CosmosDB checkpointer. Every invocation with the same `thread_id` will share conversation history — the agent "remembers" what was said before.

In [10]:
from langchain.agents import create_agent

agent = create_agent(model, checkpointer=checkpointer)

### 4. Multi-Turn Conversation (Same Thread)

Now let's test short-term memory. We send two messages on the same `thread_id`:
1. **"Hi! I'm Alice."** — introduce ourselves
2. **"What's my name?"** — the agent should remember "Alice" from the previous message

All conversation state is persisted in CosmosDB between invocations.

In [14]:
from langchain_core.messages import HumanMessage

# Use a unique thread_id — all messages with the same thread_id share memory
config = {"configurable": {"thread_id": "demo-thread-1"}}

response = agent.invoke(
    {"messages": [HumanMessage(content="Hi! I'm Mohamed and I live in Tunis.")]},
    config=config,
)

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

Hi! I'm Mohamed and I live in Tunis.
================================== Ai Message ==================================

Hi Mohamed! Nice to meet you — and hello to Tunis too. How can I help you today?
================================ Human Message =================================

Hi! I'm Mohamed and I live in Tunis.
================================== Ai Message ==================================

Hi Mohamed! Nice to meet you. Tunis is a great city. What can I help you with today?


### Turn 2: Test Memory Recall

Ask the agent a question that requires recalling information from Turn 1. Because we use the **same `thread_id`**, the CosmosDB checkpointer loads the previous conversation history and the agent can answer correctly.

In [15]:
response = agent.invoke(
    {"messages": [HumanMessage(content="What's my name and where do I live?")]},
    config=config,  # Same thread_id → agent remembers Turn 1
)

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

Hi! I'm Mohamed and I live in Tunis.
================================== Ai Message ==================================

Hi Mohamed! Nice to meet you — and hello to Tunis too. How can I help you today?
================================ Human Message =================================

Hi! I'm Mohamed and I live in Tunis.
================================== Ai Message ==================================

Hi Mohamed! Nice to meet you. Tunis is a great city. What can I help you with today?
================================ Human Message =================================

What's my name and where do I live?
================================== Ai Message ==================================

Your name is Mohamed, and you live in Tunis.


### 5. Thread Isolation

Start a **new thread** (`demo-thread-2`). The agent should **not** remember Alice's name because each thread has its own isolated conversation history in CosmosDB.

In [16]:
config_new_thread = {"configurable": {"thread_id": "demo-thread-2"}}

response = agent.invoke(
    {"messages": [HumanMessage(content="Do you know my name?")]},
    config=config_new_thread,  # Different thread_id → fresh conversation
)

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

Do you know my name?
================================== Ai Message ==================================

No—I don’t know your name unless you tell me.

If you want, you can share it, and I’ll use it in this conversation.


### 6. Resume the Original Thread

Switch back to `demo-thread-1`. The agent should still remember Alice and Amsterdam because the full conversation history is persisted in CosmosDB.

In [17]:
response = agent.invoke(
    {"messages": [HumanMessage(content="Can you remind me what we talked about earlier?")]},
    config=config,  # Back to demo-thread-1
)

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

Hi! I'm Mohamed and I live in Tunis.
================================== Ai Message ==================================

Hi Mohamed! Nice to meet you — and hello to Tunis too. How can I help you today?
================================ Human Message =================================

Hi! I'm Mohamed and I live in Tunis.
================================== Ai Message ==================================

Hi Mohamed! Nice to meet you. Tunis is a great city. What can I help you with today?
================================ Human Message =================================

What's my name and where do I live?
================================== Ai Message ==================================

Your name is Mohamed, and you live in Tunis.
================================ Human Message =================================

Can you remind me what we talked about earlier?
================================== Ai Message ===========

### 7. Check the Data Saved in CosmosDB

Finally, we can inspect the CosmosDB container to see the saved conversation state. Each message and tool call is stored as a separate document, all linked by the `thread_id`.

In [18]:
# list all documents in the CosmosDB container to verify that conversation state is being saved
from azure.cosmos import CosmosClient
import json

client = CosmosClient(url=cosmosdb_endpoint, credential=cosmosdb_key)
database = client.get_database_client("langgraph-memory")
container = database.get_container_client("checkpoints")

for item in container.read_all_items():
    print(json.dumps(item, indent=True))

# show the messages unencrypted in CosmosDB


{
 "partition_key": "checkpoint$demo-thread-1$$",
 "id": "checkpoint$demo-thread-1$$1f162953-a77e-6b47-bfff-b3b9eb37ba7e",
 "thread_id": "demo-thread-1",
 "checkpoint": "h6F2BKJ0c9kgMjAyNi0wNi0wN1QxNzoyMDo0OS41NzgyNjUrMDA6MDCiaWTZJDFmMTYyOTUzLWE3N2UtNmI0Ny1iZmZmLWIzYjllYjM3YmE3Za5jaGFubmVsX3ZhbHVlc4GpX19zdGFydF9fgahtZXNzYWdlc5HHqgWUvWxhbmdjaGFpbl9jb3JlLm1lc3NhZ2VzLmh1bWFurEh1bWFuTWVzc2FnZYanY29udGVudNkkSGkhIEknbSBNb2hhbWVkIGFuZCBJIGxpdmUgaW4gVHVuaXMusWFkZGl0aW9uYWxfa3dhcmdzgLFyZXNwb25zZV9tZXRhZGF0YYCkdHlwZaVodW1hbqRuYW1lwKJpZMCzbW9kZWxfdmFsaWRhdGVfanNvbrBjaGFubmVsX3ZlcnNpb25zgalfX3N0YXJ0X18BrXZlcnNpb25zX3NlZW6BqV9faW5wdXRfX4CwdXBkYXRlZF9jaGFubmVsc5GpX19zdGFydF9f",
 "type": "msgpack",
 "metadata": [
  "msgpack",
  "g6Zzb3VyY2WlaW5wdXSkc3RlcP+ncGFyZW50c4A="
 ],
 "parent_checkpoint_id": "",
 "_rid": "bi8GAOIV23UBAAAAAAAAAA==",
 "_self": "dbs/bi8GAA==/colls/bi8GAOIV23U=/docs/bi8GAOIV23UBAAAAAAAAAA==/",
 "_etag": "\"1e008bc4-0000-4700-0000-6a25a8710000\"",
 "_attachments": "attachments/",
 

### 8. Managing Long Conversations

With short-term memory enabled, long conversations can exceed the LLM’s context window. 

Common solutions are:

* **Trim messages:** Remove first or last N messages (before calling LLM)
* **Delete messages:** Delete messages from LangGraph state permanently
* **Summarize messages:** Summarize earlier messages in the history and replace them with a summary
* **Custom strategies:** Custom strategies (e.g., message filtering, etc.)

This allows the agent to keep track of the conversation without exceeding the LLM’s context window.

#### Summarize messages

The problem with trimming or removing messages, as shown above, is that you may lose information from culling of the message queue. Because of this, some applications benefit from a more sophisticated approach of summarizing the message history using a chat model.

![](./images/summarize_messages.png)

To summarize message history in an agent, use the built-in `SummarizationMiddleware`:

In [20]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.runnables import RunnableConfig


checkpointer = InMemorySaver()

agent = create_agent(
    model=model,
    tools=[],
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=("tokens", 4000),
            keep=("messages", 20)
        )
    ],
    checkpointer=checkpointer,
)

config: RunnableConfig = {"configurable": {"thread_id": "1"}}

agent.invoke({"messages": "hi, my name is Ali"}, config)
agent.invoke({"messages": "write a short poem about cats"}, config)
agent.invoke({"messages": "now do the same but for dogs"}, config)

final_response = agent.invoke({"messages": "what's my name?"}, config)

final_response["messages"][-1].pretty_print()

================================== Ai Message ==================================

Your name is Ali.


### More Resources

- [LangGraph Persistence](https://docs.langchain.com/oss/python/langgraph/persistence)

- [LangGraph Add Memory](https://docs.langchain.com/oss/python/langgraph/add-memory)- [Azure Cosmos DB documentation](https://learn.microsoft.com/azure/cosmos-db/)

- [Customizing Agent Memory](https://docs.langchain.com/oss/python/langchain/short-term-memory#customizing-agent-memory)- [langchain-azure-cosmosdb](https://pypi.org/project/langchain-azure-cosmosdb/)